In [1]:
from pathlib import Path
import os, IPython
pwd = os.getcwd()

def validate_results(metrics_dir: str = "metrics/"):
    """
    Run validate_results.py inside the current notebook kernel so the
    confusion-matrix and ROC figures appear inline.
    """
    metrics_dir = Path(metrics_dir).expanduser().resolve()

    if not metrics_dir.exists():
        raise FileNotFoundError(f"{metrics_dir} does not exist")

    notebook = IPython.get_ipython()
    notebook.run_line_magic(
        "run",
        f"{pwd}/validate_results.py --metrics-dir {metrics_dir}"
    )


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from datetime import datetime

# Load MNIST dataset
(_, _), (x_test, y_test) = mnist.load_data()
x_test = x_test.astype("float32") / 255.0
x_test = np.expand_dims(x_test, -1)  # shape: (num_samples, 28, 28, 1)


# Load model
model = tf.keras.models.load_model("models/cnn_model.h5", compile=False)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

2025-07-28 19:17:01.205800: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1753719426.423788   55933 gpu_device.cc:2398] Ignoring visible gpu device (device: 1, name: AMD Radeon Graphics, pci bus id: 0000:13:00.0) with AMDGPU version : gfx1036. The supported AMDGPU versions are gfx900, gfx906, gfx908, gfx90a, gfx942, gfx950, gfx1030, gfx1100, gfx1101, gfx1102, gfx1200, gfx1201.
I0000 00:00:1753719426.698258   55933 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14960 MB memory:  -> device: 0, name: AMD Radeon RX 7800 XT, pci bus id: 0000:03:00.0


In [3]:
model.summary()

Model: "keras_baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_3 (InputLayer)            │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_0 (Conv2D)                 │ (None, 26, 26, 16)     │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_conv_0 (BatchNormalization)  │ (None, 26, 26, 16)     │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_act_0 (Activation)         │ (None, 26, 26, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_0 (MaxPooling2D)           │ (None, 13, 13, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1 (Conv2D)                 │ (None, 11, 11, 24)     │         3,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_conv_1 (BatchNormalization)  │ (None, 11, 11, 24)     │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_act_1 (Activation)         │ (None, 11, 11, 24)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling2D)           │ (None, 5, 5, 24)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 600)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_0 (Dense)                 │ (None, 42)             │        25,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_dense_0 (BatchNormalization) │ (None, 42)             │           168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_act_0 (Activation)        │ (None, 42)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_dense (Dense)            │ (None, 10)             │           430 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_softmax (Activation)     │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,558 (115.46 KB)

 Trainable params: 29,394 (114.82 KB)

 Non-trainable params: 164 (656.00 B)

In [4]:
def benchmark(model, device_name, x_data, batch_size=1):
    with tf.device(device_name):
        # Rebuild the model under this device context
        model_copy = tf.keras.models.clone_model(model)
        model_copy.set_weights(model.get_weights())
        model_copy.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

        # Warm-up run
        _ = model_copy.predict(x_data[:batch_size])

        # Time measurement
        time_start = datetime.now()
        _ = model_copy.predict(x_data, batch_size=batch_size)
        time_end = datetime.now()

        # Compute performance
        delta = (time_end - time_start).total_seconds()
        latency = delta / len(x_data)
        throughput = len(x_data) / delta

        print(f"[{device_name}]")
        print(f"  Total time     : {delta:.4f} s")
        print(f"  Latency/sample : {latency*1000:.4f} ms")
        print(f"  Throughput     : {throughput:.2f} samples/sec")
        print("-" * 40)

In [5]:

# Run on CPU
benchmark(model, "/CPU:0", x_test)

# Run on GPU if available
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    benchmark(model, "/GPU:0", x_test)
else:
    print("No GPU detected.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step
  129/10000 ━━━━━━━━━━━━━━━━━━━━ 3s 393us/step

I0000 00:00:1753719447.729770   56052 service.cc:148] XLA service 0x7280b8074820 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1753719447.729794   56052 service.cc:156]   StreamExecutor device (0): Host, Default Version
2025-07-28 19:17:27.740320: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1753719447.818845   56052 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


10000/10000 ━━━━━━━━━━━━━━━━━━━━ 4s 377us/step
[/CPU:0]
  Total time     : 4.7292 s
  Latency/sample : 0.4729 ms
  Throughput     : 2114.51 samples/sec
----------------------------------------


I0000 00:00:1753719452.664177   56050 service.cc:148] XLA service 0x7280cc00bf30 initialized for platform ROCM (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1753719452.664188   56050 service.cc:156]   StreamExecutor device (0): AMD Radeon RX 7800 XT, AMDGPU ISA version: gfx1101


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/step
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 6s 578us/step
[/GPU:0]
  Total time     : 7.5527 s
  Latency/sample : 0.7553 ms
  Throughput     : 1324.04 samples/sec
----------------------------------------


In [6]:
# Convert the model to pytorch
import torch
from tensorflow.keras.models import load_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"{torch.cuda.is_available()}")


Using device: cpu
False


In [ ]:
validate_results("metrics/rf_analysis")  # Validate results and generate figures